In [ ]:
import warnings

warnings.filterwarnings("error", category=FutureWarning,
                        module=r"torch\.nn\.modules\.module")


In [ ]:
import subprocess, sys, time, json, statistics, random
import pandas as pd
from pathlib import Path

################################################################################
################################################################################
BEST_CFG = {
    "Cornell"     : {"optimizer":"adam","epochs":20,"lr_scale":0.1},
    "Dermatology" : {"optimizer":"adam","epochs":5, "lr_scale":2.0},
    "HHAR"        : {"optimizer":"adam","epochs":20,"lr_scale":0.1},
    "ISOLET"      : {"optimizer":"adam","epochs":30,"lr_scale":5.0},
    "ORL"         : {"optimizer":"adam","epochs":30,"lr_scale":0.5},
    "USPS"        : {"optimizer":"adam","epochs":20,"lr_scale":0.5},
    "Vehicle"     : {"optimizer":"adam","epochs":5, "lr_scale":2.0},
}

EPSILONS      = [1.0, 2.0, 4.0, 8.0]   
N_REPEATS     = 5                      
RESULTS_DIR   = Path("exp_results")   
RESULTS_DIR.mkdir(exist_ok=True)
SEED_BASE     = 2025            
################################################################################


def run_experiment(dataset, optimizer, epochs, lr_scale, eps=4.0):
    """Run a single experiment with given parameters and capture the output"""
    command = [
        "python", "main-opacus.py",
        "--data", dataset,
        "--optimizer", optimizer,
        "--num_epoch", str(epochs),
        "--lr_scale", str(lr_scale),
        "--eps", str(eps)
    ]
    
    print(f"Running: {' '.join(command)}")
    
    # Run the command and capture output
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = process.communicate()
    
    # Print the output for monitoring
    print(stdout)
    if stderr:
        print(f"Error: {stderr}", file=sys.stderr)
    
    # Extract test accuracy from the output
    lines = stdout.split('\n')
    final_accuracy = None
    final_epsilon = None
    
    for line in lines:
        if "Test Accuracy:" in line:
            try:
                final_accuracy = float(line.split("Test Accuracy:")[1].strip())
            except:
                pass
        if "Final Privacy Guarantee:" in line:
            try:
                final_epsilon = float(line.split("ε =")[1].split(",")[0].strip())
            except:
                pass
    
    print(f"Extracted test accuracy: {final_accuracy}")
    print(f"Extracted final epsilon: {final_epsilon}")
    
    return {
        "dataset": dataset,
        "optimizer": optimizer,
        "epochs": epochs,
        "lr_scale": lr_scale,
        "epsilon": eps if final_epsilon is None else final_epsilon,
        "test_accuracy": final_accuracy
    }

def hyperparameter_search(datasets, optimizer, epochs_range, lr_scales, eps=4.0):
    """Perform hyperparameter search and save results to CSV"""
    results = []
    
    total_experiments = len(datasets) * len(epochs_range) * len(lr_scales)
    completed = 0
    
    for dataset, epochs, lr_scale in product(datasets, epochs_range, lr_scales):
        try:
            print("\n" + "="*80)
            print(f"Progress: {completed}/{total_experiments} experiments completed")
            print(f"Current experiment: dataset={dataset}, optimizer={optimizer}, epochs={epochs}, lr_scale={lr_scale}")
            print("="*80 + "\n")
            
            result = run_experiment(dataset, optimizer, epochs, lr_scale, eps)
            results.append(result)
            
            # Save intermediate results to CSV
            pd.DataFrame(results).to_csv(f"hyperparameter_search_{optimizer}_eps{eps}.csv", index=False)
            
            # Print current results table
            df = pd.DataFrame(results)
            print("\nCurrent Results:")
            print(df.to_string())
            
            completed += 1
            
            # Add a small delay to avoid overwhelming the system
            time.sleep(1)
            
        except Exception as e:
            print(f"Error running experiment: {e}")
    
    return results

def run_once(dataset, eps, optim, epochs, lr_scale, seed):
    cmd = [
        "python", "main-opacus.py",
        "--data", dataset,
        "--optimizer", optim,
        "--num_epoch", str(epochs),
        "--lr_scale", str(lr_scale),
        "--eps", str(eps),
    ]
    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = proc.communicate()
    if err:
        print(err, file=sys.stderr)

    acc = None
    for line in out.splitlines():
        if "Test Accuracy:" in line:
            acc = float(line.split("Test Accuracy:")[1].strip())
            break
    if acc is None:
        raise RuntimeError(f"Accuracy not found in log for {dataset}, eps={eps}, seed={seed}")
    return acc

def run_all():
    rows = []
    for dataset, cfg in BEST_CFG.items():
        for eps in EPSILONS:
            acc_list = []
            for rep in range(N_REPEATS):
                seed = SEED_BASE + rep   
                acc = run_once(dataset, eps, cfg["optimizer"], cfg["epochs"], cfg["lr_scale"], seed)
                acc_list.append(acc)
            mean = statistics.mean(acc_list)
            std  = statistics.stdev(acc_list)
            rows.append({
                "dataset": dataset,
                "epsilon": eps,
                "mean": mean,
                "std": std,
                "latex": f"{mean:.3f}$\\pm${std:.3f}"
            })
            print(f"[{dataset:11s} | ε={eps:>4}]  {rows[-1]['latex']}")
            pd.DataFrame(rows).to_csv(RESULTS_DIR / "summary_progress.csv", index=False)
    return pd.DataFrame(rows)

if __name__ == "__main__":
    df_summary = run_all()
    out_file = RESULTS_DIR / "summary_final.csv"
    df_summary.to_csv(out_file, index=False)
    print(f"\n★ Summary → {out_file.absolute()}\n")

    def to_latex_row(ds):
        sub = df_summary[df_summary.dataset == ds].sort_values("epsilon")
        vals = " & ".join(sub["latex"].tolist())
        return f"{ds} & {vals} \\\\"

    print("============= LaTeX Rows =============")
    for d in BEST_CFG.keys():
        print(to_latex_row(d))


In [ ]:
import subprocess, sys, time, json, statistics, random
import pandas as pd
from pathlib import Path

################################################################################
################################################################################
BEST_CFG = {
    "Cornell"     : {"optimizer":"adam","epochs":20,"lr_scale":0.1},
    "Dermatology" : {"optimizer":"adam","epochs":5, "lr_scale":2.0},
    "HHAR"        : {"optimizer":"adam","epochs":20,"lr_scale":0.1},
    "ISOLET"      : {"optimizer":"adam","epochs":30,"lr_scale":5.0},
    "ORL"         : {"optimizer":"adam","epochs":30,"lr_scale":0.5},
    "USPS"        : {"optimizer":"adam","epochs":20,"lr_scale":0.5},
    "Vehicle"     : {"optimizer":"adam","epochs":5, "lr_scale":2.0},
}

EPSILONS      = [1.0, 2.0, 4.0, 8.0]  
N_REPEATS     = 5                    
RESULTS_DIR   = Path("exp_results")   
RESULTS_DIR.mkdir(exist_ok=True)
SEED_BASE     = 2025          


#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
exp_sgd_svm.py ── DP-SGD + SVM loss + LR-decay
"""

import subprocess
import sys
import time
import re
import statistics
import pandas as pd
from pathlib import Path


OPTIMIZER  = "adam"
LOSS_ARG   = []                           
EPSILONS   = [1.0, 2.0, 4.0, 8.0]
N_REPEATS  = 5
OUT_DIR    = Path("exp_results_sgd_svm")
OUT_DIR.mkdir(exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
def run_once(
    ds: str,
    eps: float,
    epc: int,
    lr: float,
    seed: int | None = None,
    use_lr_decay: bool = True,
) -> float:
    """main-opacus-decay.py"""
    cmd = [
        "python", "main-opacus-decay.py",
        "--data",       ds,
        "--optimizer",  OPTIMIZER,
        "--num_epoch",  str(epc),
        "--lr_scale",   str(lr),
        "--eps",        str(eps),
    ] + LOSS_ARG

    if use_lr_decay:
        cmd.append("--lr_decay")

    if seed is not None:
        cmd += ["--seed", str(seed)]

    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True)
    out, err = proc.communicate()
    if err:
        print(err, file=sys.stderr)

    acc = None
    for ln in out.splitlines():
        m = re.search(r"\bacc=([0-9.]+)", ln)
        if m:
            acc = float(m.group(1))  
    if acc is None:
        raise RuntimeError(f"accuracy not found for {ds}, eps={eps}")
    return acc

# ──────────────────────────────────────────────────────────────────────────────
def main() -> None:
    rows = []
    for ds, cfg in BEST_CFG.items():
        for eps in EPSILONS:
            acc_list = []
            for rep in range(N_REPEATS):
                acc = run_once(ds, eps, cfg["epochs"], cfg["lr_scale"], seed=None)
                acc_list.append(acc)

            mean = statistics.mean(acc_list)
            std  = statistics.stdev(acc_list)
            rows.append({
                "dataset": ds,
                "epsilon": eps,
                "mean":    mean,
                "std":     std,
                "latex":   f"{mean:.3f}$\\pm${std:.3f}"
            })
            pd.DataFrame(rows).to_csv(OUT_DIR / "summary_progress.csv", index=False)
            print(f"[{ds:11s} | ε={eps:>4}] {rows[-1]['latex']}")

    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / "summary_final.csv", index=False)
    print("\n▶ Summary:", OUT_DIR / "summary_final.csv")

    print("\n============= LaTeX Rows =============")
    for ds in BEST_CFG.keys():
        vals = df.query("dataset == @ds").sort_values("epsilon")["latex"].tolist()
        print(f"{ds} & " + " & ".join(vals) + r" \\")
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()
